# DATA TABLES FOR CEP UK DYNAMISM PAPER

In [22]:
# Import packages and set filepaths
import pandas as pd
import numpy as np
import altair as alt
from pathlib import Path
from pandas.api.types import CategoricalDtype
import os
import eco_style 
alt.themes.enable("report")

script_dir = Path.cwd()
import_path = script_dir.parent / "Data"
export_path = script_dir.parent / "Tables"

In [23]:
# IMPORT DATASETS
population_df = pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='population')
firm_dynamics_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='firm_dynamics')
job_flows_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='job_flows')
site_dynamics_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='site_dynamics')
cohort_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='cohort_analysis')
growth_rates_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='growth_rates')
growth_cats_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='growth_cats')
prod_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='prod')


In [58]:
# Force data types to numeric where possible, * for diclosive values automatically read the column in as a string
def convert_numeric_columns(df):
    """
    Convert all numeric columns to numeric type, replacing '*' with NaN.
    Preserves non-numeric columns (like year, category names) as they are.
    """
    df_converted = df.copy()
    
    for col in df_converted.columns:
        # Try to convert to numeric, replacing '*' and other errors with NaN
        df_converted[col] = pd.to_numeric(df_converted[col].replace('*', np.nan), errors='ignore')
    
    return df_converted

# Apply to all dataframes
population_df = convert_numeric_columns(population_df)
firm_dynamics_df = convert_numeric_columns(firm_dynamics_df)
job_flows_df = convert_numeric_columns(job_flows_df)
site_dynamics_df = convert_numeric_columns(site_dynamics_df)
cohort_df = convert_numeric_columns(cohort_df)
growth_rates_df = convert_numeric_columns(growth_rates_df)
growth_cats_df = convert_numeric_columns(growth_cats_df)
prod_df = convert_numeric_columns(prod_df)

/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_3101/2220193345.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_converted[col] = pd.to_numeric(df_converted[col].replace('*', np.nan), errors='ignore')


## **COMPOSITION OF UK FIRMS**

In [20]:
# BASIC FACTS TABLE FOR WHOLE ECONOMY
df = population_df[population_df['dimension']=='Total']

# turnover in data is in £000s (matches chart: 4.4bn * 1000 = £4.4tn)
# avg_turnover_per_employee is in £000s (97 -> £97k matches chart £100k range)
df['Total firms (m)']                   = (df['n_firms'] / 1e6).round(2)
df['Total employment (m)']              = (df['employment'] / 1e6).round(2)
df['Total turnover (£tn)']              = (df['turnover'] * 1000 / 1e12).round(2)
df['Avg turnover per employee (£000s)'] = df['avg_turnover_per_employee'].round(0).astype(int)

headline_table = df[['year',
            'Total firms (m)',
            'Total employment (m)',
            'Total turnover (£tn)',
            'Avg turnover per employee (£000s)']].reset_index()

headline_table

/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_3101/1103691155.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Total firms (m)']                   = (df['n_firms'] / 1e6).round(2)
/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_3101/1103691155.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Total employment (m)']              = (df['employment'] / 1e6).round(2)
/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_3101/1103691155.py:8: SettingWithCopyWarning: 
A v

,index,year,Total firms (m),Total employment (m),Total turnover (£tn),Avg turnover per employee (£000s)
0,39,1997,1.76,17.77,1.84,97
1,79,1998,1.92,18.64,1.79,91
2,119,1999,1.94,18.82,1.89,90
3,159,2000,1.93,18.99,1.91,92
4,199,2001,1.95,19.36,2.01,98
5,239,2002,1.95,19.88,2.08,95
6,279,2003,1.96,19.90,2.17,103
7,319,2004,2.00,19.77,2.23,104
8,359,2005,2.04,19.96,2.33,100
9,399,2006,2.08,20.17,2.42,102


In [48]:
# COMPOSITION BY FIRM SIZE

df = population_df[population_df['dimension']=='Size']

size_order = ['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)']

df_2023 = df[df['year'] == 2023].set_index('category')
df_2003 = df[df['year'] == 2003].set_index('category')

rows = []
for cat in size_order:
    firms_2023 = df_2023.loc[cat, 'n_firms']
    emp_2023   = df_2023.loc[cat, 'employment']
    firms_2003 = df_2003.loc[cat, 'n_firms']
    emp_2003   = df_2003.loc[cat, 'employment']

    rows.append({
        'Firm size (employees)': cat,
        'Firms (000s)': round(firms_2023 / 1000),        # integer
        'Empl (000s)':  round(emp_2023  / 1000),         # integer
        'Firms (%)':    round((firms_2023 - firms_2003) / firms_2003 * 100, 1),
        'Empl (%)':     round((emp_2023  - emp_2003)  / emp_2003  * 100, 1),
    })

# Total row
total_firms_2023 = df_2023['n_firms'].sum()
total_emp_2023   = df_2023['employment'].sum()
total_firms_2003 = df_2003['n_firms'].sum()
total_emp_2003   = df_2003['employment'].sum()

rows.append({
    'Firm size (employees)': 'Total',
    'Firms (000s)': round(total_firms_2023 / 1000),
    'Empl (000s)':  round(total_emp_2023  / 1000),
    'Firms (%)':    round((total_firms_2023 - total_firms_2003) / total_firms_2003 * 100, 1),
    'Empl (%)':     round((total_emp_2023  - total_emp_2003)  / total_emp_2003  * 100, 1),
})

size_composition = pd.DataFrame(rows).set_index('Firm size (employees)')
size_composition.columns = pd.MultiIndex.from_tuples([
    ('2023 Levels', 'Firms (000s)'),
    ('2023 Levels', 'Empl (000s)'),
    ('Change 2003–2023', 'Firms (%)'),
    ('Change 2003–2023', 'Empl (%)'),
])

size_composition = size_composition.reset_index()
size_composition

Firm size (employees)  2023 Levels             Change 2003–2023         
                        Firms (000s) Empl (000s)        Firms (%) Empl (%)
0           Micro (0-9)         2228        5178             28.2     16.1
1         Small (10-49)          226        4451             25.7     28.4
2       Medium (50-249)           40        3832             29.6     28.3
3          Large (250+)            8       10580             21.1     17.8
4                 Total         2502       24041             28.0     20.8

In [51]:
# COMPOSITION BY FIRM AGE

df = population_df[population_df['dimension']=='Age']


age_order = ['New (0-2 years)', 'Young (3-5 years)', 'Mature (6-10 years)', 'Old (11+ years)']

df_2023 = df[df['year'] == 2023].set_index('category')
df_2003 = df[df['year'] == 2003].set_index('category')

rows = []
for cat in age_order:
    firms_2023 = df_2023.loc[cat, 'n_firms']
    emp_2023   = df_2023.loc[cat, 'employment']
    firms_2003 = df_2003.loc[cat, 'n_firms']
    emp_2003   = df_2003.loc[cat, 'employment']

    rows.append({
        'Firm age': cat,
        'Firms (000s)': round(firms_2023 / 1000),        # integer
        'Empl (000s)':  round(emp_2023  / 1000),         # integer
        'Firms (%)':    round((firms_2023 - firms_2003) / firms_2003 * 100, 1),
        'Empl (%)':     round((emp_2023  - emp_2003)  / emp_2003  * 100, 1),
    })


# Total row
total_firms_2023 = df_2023['n_firms'].sum()
total_emp_2023   = df_2023['employment'].sum()
total_firms_2003 = df_2003['n_firms'].sum()
total_emp_2003   = df_2003['employment'].sum()

rows.append({
    'Firm age': 'Total',
    'Firms (000s)': round(total_firms_2023 / 1000),
    'Empl (000s)':  round(total_emp_2023  / 1000),
    'Firms (%)':    round((total_firms_2023 - total_firms_2003) / total_firms_2003 * 100, 1),
    'Empl (%)':     round((total_emp_2023  - total_emp_2003)  / total_emp_2003  * 100, 1),
})

age_composition = pd.DataFrame(rows).set_index('Firm age')
age_composition.columns = pd.MultiIndex.from_tuples([
    ('2023 Levels', 'Firms (000s)'),
    ('2023 Levels', 'Empl (000s)'),
    ('Change 2003–2023', 'Firms (%)'),
    ('Change 2003–2023', 'Empl (%)'),
])

age_composition = age_composition.reset_index()
age_composition

Firm age  2023 Levels             Change 2003–2023         
                       Firms (000s) Empl (000s)        Firms (%) Empl (%)
0      New (0-2 years)          696        2316             16.0    -14.7
1    Young (3-5 years)          420        2115              6.9    -23.4
2  Mature (6-10 years)          401        2386             87.5     82.1
3      Old (11+ years)          986       17223             31.6     31.4
4                Total         2502       24041             28.0     20.8

In [ ]:
# COMPOSITION BY FIRM REGION

df = population_df[population_df['dimension']=='Region']

df_2023 = df[df['year'] == 2023].set_index('category')
df_2003 = df[df['year'] == 2003].set_index('category')

region_order = df_2023['n_firms'].sort_values(ascending=False).index.tolist()

rows = []
for cat in region_order:
    firms_2023 = df_2023.loc[cat, 'n_firms']
    emp_2023   = df_2023.loc[cat, 'employment']
    firms_2003 = df_2003.loc[cat, 'n_firms']
    emp_2003   = df_2003.loc[cat, 'employment']

    rows.append({
        'Region': cat,
        'Firms (000s)': round(firms_2023 / 1000),        # integer
        'Empl (000s)':  round(emp_2023  / 1000),         # integer
        'Firms (%)':    round((firms_2023 - firms_2003) / firms_2003 * 100, 1),
        'Empl (%)':     round((emp_2023  - emp_2003)  / emp_2003  * 100, 1),
    })

# Total row
total_firms_2023 = df_2023['n_firms'].sum()
total_emp_2023   = df_2023['employment'].sum()
total_firms_2003 = df_2003['n_firms'].sum()
total_emp_2003   = df_2003['employment'].sum()

rows.append({
    'Region': 'Total',
    'Firms (000s)': round(total_firms_2023 / 1000),
    'Empl (000s)':  round(total_emp_2023  / 1000),
    'Firms (%)':    round((total_firms_2023 - total_firms_2003) / total_firms_2003 * 100, 1),
    'Empl (%)':     round((total_emp_2023  - total_emp_2003)  / total_emp_2003  * 100, 1),
})

region_composition = pd.DataFrame(rows).set_index('Region')
region_composition.columns = pd.MultiIndex.from_tuples([
    ('2023 Levels', 'Firms (000s)'),
    ('2023 Levels', 'Empl (000s)'),
    ('Change 2003–2023', 'Firms (%)'),
    ('Change 2003–2023', 'Empl (%)'),
])

region_composition = region_composition.reset_index()
region_composition

Region  2023 Levels             Change 2003–2023  \
                             Firms (000s) Empl (000s)        Firms (%)   
0                     London          478        4961             60.8   
1                 South East          372        3271             21.4   
2            East Of England          247        2540             24.6   
3                 North West          242        2358             26.9   
4                 South West          217        1759             16.3   
5              West Midlands          201        2086             23.9   
6   Yorkshire and The Humber          177        1860             25.4   
7              East Midlands          167        1802             23.7   
8                   Scotland          160        1441             16.1   
9                      Wales          100         770             13.8   
10          Northern Ireland           74         524             22.9   
11                North East           67         670             29.2   
12                     Total         2502       24041             28.0   

             
   Empl (%)  
0      34.3  
1      11.6  
2      26.3  
3      19.6  
4      25.0  
5      17.4  
6      23.6  
7      10.5  
8      12.8  
9      23.9  
10     17.3  
11      7.7  
12     20.8

In [53]:
# COMPOSITION BY FIRM SECTOR

df = population_df[population_df['dimension']=='Sector']

df_2023 = df[df['year'] == 2023].set_index('category')
df_2003 = df[df['year'] == 2003].set_index('category')

sector_order = df_2023['n_firms'].sort_values(ascending=False).index.tolist()

rows = []
for cat in sector_order:
    firms_2023 = df_2023.loc[cat, 'n_firms']
    emp_2023   = df_2023.loc[cat, 'employment']
    firms_2003 = df_2003.loc[cat, 'n_firms']
    emp_2003   = df_2003.loc[cat, 'employment']

    rows.append({
        'Sector': cat,
        'Firms (000s)': round(firms_2023 / 1000),        # integer
        'Empl (000s)':  round(emp_2023  / 1000),         # integer
        'Firms (%)':    round((firms_2023 - firms_2003) / firms_2003 * 100, 1),
        'Empl (%)':     round((emp_2023  - emp_2003)  / emp_2003  * 100, 1),
    })


# Total row
total_firms_2023 = df_2023['n_firms'].sum()
total_emp_2023   = df_2023['employment'].sum()
total_firms_2003 = df_2003['n_firms'].sum()
total_emp_2003   = df_2003['employment'].sum()

rows.append({
    'Sector': 'Total',
    'Firms (000s)': round(total_firms_2023 / 1000),
    'Empl (000s)':  round(total_emp_2023  / 1000),
    'Firms (%)':    round((total_firms_2023 - total_firms_2003) / total_firms_2003 * 100, 1),
    'Empl (%)':     round((total_emp_2023  - total_emp_2003)  / total_emp_2003  * 100, 1),
})

sector_composition = pd.DataFrame(rows).set_index('Sector')
sector_composition.columns = pd.MultiIndex.from_tuples([
    ('2023 Levels', 'Firms (000s)'),
    ('2023 Levels', 'Empl (000s)'),
    ('Change 2003–2023', 'Firms (%)'),
    ('Change 2003–2023', 'Empl (%)'),
])

sector_composition = sector_composition.reset_index()
sector_composition

Sector  2023 Levels             Change 2003–2023  \
                               Firms (000s) Empl (000s)        Firms (%)   
0        Professional Services          424        2661             78.6   
1                 Construction          389        1650             58.9   
2    Business Support Services          237        2970             31.5   
3                 Retail Trade          220        3012              1.4   
4              Wholesale Trade          189        1817              4.4   
5               Other Services          187        1377              0.4   
6                  Hospitality          178        2535             26.6   
7                Manufacturing          142        2488             -9.9   
8       IT & Computer Services          136         865             17.7   
9                  Agriculture          134         483             -5.5   
10       Transport & Logistics          130        1476             82.8   
11  Other Information Services           57         545             73.4   
12                 Social care           52        1739             45.4   
13                   Utilities           16         336            301.8   
14    Other Primary Industries           11          87             18.7   
15                       Total         2502       24041             28.0   

             
   Empl (%)  
0      80.6  
1      20.4  
2      46.5  
3       2.4  
4       7.6  
5      18.1  
6      49.0  
7     -30.7  
8      54.5  
9      22.7  
10     21.5  
11     -4.6  
12     89.4  
13     72.5  
14      0.4  
15     20.8

In [54]:
with pd.ExcelWriter(export_path / 'BSD_firm_composition.xlsx', engine='openpyxl') as writer:
    headline_table.to_excel(writer, sheet_name='headline', index=False)
    size_composition.to_excel(writer, sheet_name='size', index=True)
    age_composition.to_excel(writer, sheet_name='age', index=True)
    region_composition.to_excel(writer, sheet_name='region', index=True)
    sector_composition.to_excel(writer, sheet_name='sector', index=True)

## **ENTRY AND EXIT RATES**

In [85]:
# Calculate entry and exit rates

firm_dynamics_df = firm_dynamics_df.sort_values(['category','dimension','year'])

firm_dynamics_df['total_firms_lag'] = firm_dynamics_df.groupby(['category','dimension'])['n_firms'].shift(1)

firm_dynamics_df['entry_rate'] = (firm_dynamics_df['n_entrants'] + firm_dynamics_df['n_entry_and_exit']) / firm_dynamics_df['total_firms_lag']
firm_dynamics_df['exit_rate'] = (firm_dynamics_df['n_exiters'] + firm_dynamics_df['n_entry_and_exit']) / firm_dynamics_df['total_firms_lag']

In [86]:
# TOTAL ENTRY AND EXIT RATES EACH YEAR

entry_exit_total = (
    firm_dynamics_df[(firm_dynamics_df['dimension'] == 'Total') & (firm_dynamics_df['year'] > 1998)][['year', 'entry_rate', 'exit_rate']]
    .assign(entry_rate=lambda x: (x['entry_rate'] * 100).round(1),
            exit_rate= lambda x: (x['exit_rate']  * 100).round(1),
            net_entry_rate= lambda x: (x['entry_rate'] - x['exit_rate']).round(1))
    .rename(columns={'year': 'Year', 'entry_rate': 'Entry rate', 'exit_rate': 'Exit rate', 'net_entry_rate': 'Net entry rate'})
    .sort_values('Year')
    .reset_index(drop=True)
)
entry_exit_total

,Year,Entry rate,Exit rate,Net entry rate
0,1999,12.3,14.0,-1.7
1,2000,12.6,11.7,0.9
2,2001,12.5,12.9,-0.4
3,2002,12.4,13.2,-0.8
4,2003,12.9,13.8,-0.9
5,2004,15.5,13.7,1.8
6,2005,14.5,12.9,1.6
7,2006,13.7,12.1,1.6
8,2007,14.4,16.7,-2.3
9,2008,14.8,14.8,0.0


In [87]:
# ENTRY AND EXIT RATES BY FIRM SIZE

df = firm_dynamics_df[firm_dynamics_df['dimension'] == 'Size']
df = df[df['year']>1998]

size_order = ['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)']
years = sorted(df['year'].unique())

rows = []
for year in years:
    df_year = df[df['year'] == year].set_index('category')
    
    row = {'Year': year}
    for size in size_order:
        row[f'{size}__entry_rate'] = round(df_year.loc[size, 'entry_rate'] * 100, 1)
        row[f'{size}__exit_rate']  = round(df_year.loc[size, 'exit_rate']  * 100, 1)
    
    rows.append(row)

entry_exit_size = pd.DataFrame(rows).set_index('Year')

entry_exit_size.columns = pd.MultiIndex.from_tuples([
    (size, metric)
    for size in size_order
    for metric in ['Entry rate', 'Exit rate']
])

entry_exit_size = entry_exit_size.reset_index()
entry_exit_size

Year Micro (0-9)           Small (10-49)           Medium (50-249)  \
          Entry rate Exit rate    Entry rate Exit rate      Entry rate   
0   1999        13.3      14.6           4.5       9.1             4.5   
1   2000        13.5      12.2           4.9       7.7             5.1   
2   2001        13.4      13.5           5.2       7.9             4.7   
3   2002        13.3      13.8           5.6       8.9             4.4   
4   2003        13.9      14.6           5.2       7.7             3.7   
5   2004        16.8      14.6           5.1       7.0             4.0   
6   2005        15.6      13.6           5.0       7.0             3.8   
7   2006        14.7      12.7           5.1       7.1             3.6   
8   2007        15.5      17.6           5.0       8.6             3.1   
9   2008        15.9      15.7           5.3       7.0             4.3   
10  2009        11.8      14.4           4.3       7.4             3.0   
11  2010        10.6      13.8           3.8       6.3             2.7   
12  2011        10.9      11.9           4.2       5.9             2.8   
13  2012        14.9      14.3           4.8       6.4             2.8   
14  2013        13.2      13.2           4.3       6.1             2.7   
15  2014        17.2      13.7           5.4       5.6             3.6   
16  2015        15.9      12.8           5.1       5.6             3.9   
17  2016        15.8      13.5           4.5       5.3             3.6   
18  2017        16.4      15.5           5.2       5.6             3.4   
19  2018        13.7      13.4           4.6       5.6             3.4   
20  2019        14.6      14.3           4.7       5.6             3.0   
21  2020        14.3      13.9           4.7       4.0             3.0   
22  2021        13.4      15.6           3.6       3.9             2.4   
23  2022        14.0      15.3           5.2       5.3             2.7   

             Large (250+)            
   Exit rate   Entry rate Exit rate  
0        8.1          4.3       7.3  
1        8.1          4.3       8.1  
2        7.9          5.0       7.4  
3        9.2          4.2       8.0  
4        6.8          4.0       7.5  
5        7.0          4.2       7.2  
6        6.8          2.7       6.2  
7        7.0          2.1       6.0  
8        9.0          2.1       7.1  
9        6.7          2.5       6.1  
10       6.3          2.1       4.8  
11       5.5          1.4       4.0  
12       4.9          1.3       3.3  
13       4.9          1.5       3.3  
14       4.8          1.0       3.5  
15       4.8          2.9       3.5  
16       4.6          4.4       4.0  
17       4.1          3.4       3.8  
18       4.2          3.1       3.4  
19       4.2          3.6       4.2  
20       3.8          3.5       3.3  
21       3.1          2.4       2.2  
22       2.8          1.6       2.3  
23       3.4          1.8       3.2

In [89]:
age_order = ['New (0-2 years)', 'Young (3-5 years)', 'Mature (6-10 years)', 'Old (11+ years)']

exit_age = (
    firm_dynamics_df[(firm_dynamics_df['dimension'] == 'Age') & (firm_dynamics_df['year'] > 1998)][['year', 'category', 'exit_rate']]
    .assign(exit_rate=lambda x: (x['exit_rate'] * 100).round(1))
    .pivot(index='year', columns='category', values='exit_rate')
    [age_order]
    .rename_axis('Year')
    .rename_axis(None, axis=1)
    .reset_index()
)
exit_age

,Year,New (0-2 years),Young (3-5 years),Mature (6-10 years),Old (11+ years)
0,1999,20.6,16.7,13.8,6.7
1,2000,21.0,9.0,9.8,6.3
2,2001,17.0,21.3,11.5,6.9
3,2002,19.5,14.8,12.4,7.9
4,2003,20.3,18.8,9.2,8.3
5,2004,21.4,12.9,15.9,7.5
6,2005,19.5,15.1,10.7,7.1
7,2006,17.8,13.8,10.4,6.9
8,2007,24.3,21.0,15.4,8.2
9,2008,22.7,17.2,10.2,7.9


In [90]:
# ENTRY AND EXIT RATES BY FIRM PRODUCTIVITY

df = firm_dynamics_df[firm_dynamics_df['dimension'] == 'Productivity']
df = df[df['year']>1998]

prod_order = ['Laggards (<P10)','Low-Median (P10-P50)','High-Median (P50-P90)','Frontier (P90+)']
years = sorted(df['year'].unique())

rows = []
for year in years:
    df_year = df[df['year'] == year].set_index('category')
    
    row = {'Year': year}
    for prod in prod_order:
        row[f'{prod}__entry_rate'] = round(df_year.loc[prod, 'entry_rate'] * 100, 1)
        row[f'{prod}__exit_rate']  = round(df_year.loc[prod, 'exit_rate']  * 100, 1)
    
    rows.append(row)

entry_exit_prod = pd.DataFrame(rows).set_index('Year')

entry_exit_prod.columns = pd.MultiIndex.from_tuples([
    (prod, metric)
    for prod in prod_order
    for metric in ['Entry rate', 'Exit rate']
])

entry_exit_prod = entry_exit_prod.reset_index()
entry_exit_prod

Year Laggards (<P10)           Low-Median (P10-P50)            \
              Entry rate Exit rate           Entry rate Exit rate   
0   1999             6.7      19.8                 11.8      12.4   
1   2000             5.6      14.5                 14.2      12.7   
2   2001             6.2      20.0                 12.2      12.6   
3   2002             5.1      22.7                 12.9      12.7   
4   2003             5.4      19.5                 13.5      13.8   
5   2004             7.3      15.2                 17.8      14.0   
6   2005             7.4      15.3                 15.9      13.3   
7   2006             7.7      15.7                 15.7      12.8   
8   2007             8.9      21.1                 15.3      17.5   
9   2008             8.7      18.3                 14.4      15.3   
10  2009             6.3      18.0                 10.5      13.3   
11  2010             6.4      18.0                  9.5      12.9   
12  2011             7.3      17.1                  8.5      10.3   
13  2012             7.9      19.3                 14.1      13.7   
14  2013             7.9      18.3                 11.9      12.1   
15  2014            10.3      17.8                 17.2      13.2   
16  2015            10.5      17.0                 15.6      12.0   
17  2016            10.6      17.3                 15.6      13.0   
18  2017            11.0      18.9                 16.4      15.5   
19  2018             9.4      17.3                 12.7      13.0   
20  2019            10.8      20.3                 14.2      14.0   
21  2020            13.3      18.4                 13.7      12.7   
22  2021            14.1      22.2                 12.6      14.4   
23  2022            13.6      22.0                 12.0      13.7   

   High-Median (P50-P90)           Frontier (P90+)            
              Entry rate Exit rate      Entry rate Exit rate  
0                   15.7      15.5             7.3       8.2  
1                   14.3      10.9             7.0       7.1  
2                   15.9      12.6             7.0       8.2  
3                   15.2      12.2             7.7       8.6  
4                   15.8      13.4             8.1       9.2  
5                   17.1      14.4             8.5       8.3  
6                   16.8      13.0             7.5       8.3  
7                   14.9      11.4             7.5       8.0  
8                   16.8      16.1             8.3      10.1  
9                   19.1      14.6             7.8       9.1  
10                  14.6      13.7             5.6       9.8  
11                  12.4      12.8             5.1       9.3  
12                  13.9      11.6             5.9       7.8  
13                  16.6      12.5             7.1      10.1  
14                  15.3      12.2             6.0       8.8  
15                  17.8      12.0             8.5       9.3  
16                  16.9      11.5             7.3       8.5  
17                  16.5      11.8             6.4       8.6  
18                  17.3      13.4             6.3       8.6  
19                  15.5      11.8             6.5       8.3  
20                  15.2      11.9             6.8       8.5  
21                  14.7      12.6             6.0       9.2  
22                  13.4      13.5             5.1       8.8  
23                  15.7      14.0             7.4       9.2

In [91]:
# ENTRY AND EXIT RATES BY SECTOR - PERIOD AVERAGES, ORDERED BY NET CHURN

periods = {
    '2000-2007': (2000, 2007),
    '2008-2015': (2008, 2015),
    '2016-2022': (2016, 2022),
}

df = firm_dynamics_df[firm_dynamics_df['dimension'] == 'Sector']

# Calculate average net churn across all years to determine ordering
sector_churn = (
    df.groupby('category')[['entry_rate', 'exit_rate']]
    .mean()
    .assign(net_churn=lambda x: x['entry_rate'] + x['exit_rate'])
    .sort_values('net_churn', ascending=False)
)
sector_order = sector_churn.index.tolist()

rows = []
for sector in sector_order:
    df_sector = df[df['category'] == sector]
    
    row = {'Sector': sector}
    for period_label, (year_start, year_end) in periods.items():
        df_period = df_sector[(df_sector['year'] >= year_start) & (df_sector['year'] <= year_end)]
        row[f'{period_label}__entry_rate'] = round(df_period['entry_rate'].mean() * 100, 1)
        row[f'{period_label}__exit_rate']  = round(df_period['exit_rate'].mean()  * 100, 1)
    
    rows.append(row)

entry_exit_sector = pd.DataFrame(rows).set_index('Sector')

entry_exit_sector.columns = pd.MultiIndex.from_tuples([
    (period, metric)
    for period in periods.keys()
    for metric in ['Entry rate', 'Exit rate']
])

entry_exit_sector = entry_exit_sector.reset_index()
entry_exit_sector

Sector  2000-2007            2008-2015            \
                               Entry rate Exit rate Entry rate Exit rate   
0    Business Support Services       23.2      18.7       15.8      16.4   
1                    Utilities       33.3      15.8       20.8      15.1   
2       IT & Computer Services       19.1      20.4       17.4      14.5   
3                  Hospitality       19.5      19.0       15.3      16.7   
4        Transport & Logistics       13.8      13.6       13.9      14.0   
5        Professional Services       16.0      13.5       17.2      14.2   
6   Other Information Services       14.8      13.7       16.2      14.4   
7                 Construction       14.5      13.0       12.6      13.9   
8                 Retail Trade       11.3      13.2       11.8      12.7   
9               Other Services       11.7      11.5        9.9      11.4   
10             Wholesale Trade        9.4      10.8        9.6      10.7   
11               Manufacturing        8.8      11.1        9.2      10.6   
12    Other Primary Industries        9.8      11.9        9.7      10.5   
13                 Social care        8.9       9.4       11.5       9.5   
14                 Agriculture        3.2       6.9        3.2       5.0   

    2016-2022            
   Entry rate Exit rate  
0        18.5      18.1  
1        13.6      10.7  
2        13.1      15.2  
3        17.1      16.2  
4        26.1      22.9  
5        12.9      14.0  
6        13.3      12.4  
7        14.5      12.5  
8        15.1      14.6  
9        10.3      10.5  
10       10.2      10.3  
11       10.4      10.7  
12        8.9       9.6  
13        8.8       9.5  
14        2.9       5.1

In [ ]:
# ENTRY AND EXIT RATES BY REGION - PERIOD AVERAGES, ORDERED BY NET CHURN

periods = {
    '2000-2007': (2000, 2007),
    '2008-2015': (2008, 2015),
    '2016-2022': (2016, 2022),
}

df = firm_dynamics_df[firm_dynamics_df['dimension'] == 'Region']

# Calculate average net churn across all years to determine ordering
region_churn = (
    df.groupby('category')[['entry_rate', 'exit_rate']]
    .mean()
    .assign(net_churn=lambda x: x['entry_rate'] + x['exit_rate'])
    .sort_values('net_churn', ascending=False)
)
region_order = region_churn.index.tolist()

rows = []
for region in region_order:
    df_region = df[df['category'] == region]
    
    row = {'Region': region}
    for period_label, (year_start, year_end) in periods.items():
        df_period = df_region[(df_region['year'] >= year_start) & (df_region['year'] <= year_end)]
        row[f'{period_label}__entry_rate'] = round(df_period['entry_rate'].mean() * 100, 1)
        row[f'{period_label}__exit_rate']  = round(df_period['exit_rate'].mean()  * 100, 1)
    
    rows.append(row)

entry_exit_region = pd.DataFrame(rows).set_index('Region')

entry_exit_region.columns = pd.MultiIndex.from_tuples([
    (period, metric)
    for period in periods.keys()
    for metric in ['Entry rate', 'Exit rate']
])

entry_exit_region = entry_exit_region.reset_index()
entry_exit_region

Region  2000-2007            2008-2015            \
                             Entry rate Exit rate Entry rate Exit rate   
0                     London       16.3      15.9       17.8      15.4   
1                 North West       14.5      13.9       13.7      14.2   
2                 North East       13.3      13.4       13.2      13.1   
3                 South East       14.8      14.0       12.1      13.0   
4              West Midlands       12.7      12.7       11.9      12.4   
5   Yorkshire and The Humber       13.5      13.3       12.1      12.6   
6            East Of England       12.9      12.9       12.1      12.3   
7              East Midlands       12.8      12.8       12.0      12.3   
8                   Scotland       11.6      12.2       11.9      11.8   
9                 South West       12.3      12.4       10.5      11.3   
10                     Wales       11.5      12.0       10.1      11.4   
11          Northern Ireland        9.0       9.1        7.6       9.9   

    2016-2022            
   Entry rate Exit rate  
0        17.4      15.9  
1        14.7      14.5  
2        13.3      13.4  
3        12.4      12.9  
4        14.5      14.3  
5        12.7      12.7  
6        13.0      13.3  
7        13.1      13.3  
8        11.4      12.4  
9        10.6      11.3  
10       11.7      12.3  
11        8.9       9.1

In [104]:
# NET ENTRY RATE BY REGION - PERIOD AVERAGES, ORDERED BY NET CHURN

periods = {
    '2000-2007': (2000, 2007),
    '2008-2015': (2008, 2015),
    '2016-2022': (2016, 2022),
}

df = firm_dynamics_df[firm_dynamics_df['dimension'] == 'Region']
df['net_entry_rate'] = df['entry_rate'] - df['exit_rate']

# Calculate average net churn across all years to determine ordering
region_churn = (
    df.groupby('category')[['net_entry_rate']]
    .mean()
    .sort_values('net_entry_rate', ascending=False)
)
region_order = region_churn.index.tolist()

rows = []
for region in region_order:
    df_region = df[df['category'] == region]
    
    row = {'Region': region}
    for period_label, (year_start, year_end) in periods.items():
        df_period = df_region[(df_region['year'] >= year_start) & (df_region['year'] <= year_end)]
        row[f'{period_label}__net_entry_rate'] = round(df_period['net_entry_rate'].mean() * 100, 1)
    
    rows.append(row)

net_entry_region = pd.DataFrame(rows).set_index('Region')

net_entry_region.columns = pd.MultiIndex.from_tuples([
    (period, metric)
    for period in periods.keys()
    for metric in ['Net entry rate']
])

net_entry_region = net_entry_region.reset_index()
net_entry_region

/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_3101/1715067058.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['net_entry_rate'] = df['entry_rate'] - df['exit_rate']


,Region,2000-2007,2008-2015,2016-2022
,,Net entry rate,Net entry rate,Net entry rate
0,London,0.3,2.4,1.5
1,North West,0.6,-0.5,0.2
2,North East,-0.1,0.2,-0.0
3,South East,0.8,-0.8,-0.4
4,West Midlands,-0.0,-0.5,0.2
5,East Of England,0.0,-0.2,-0.3
6,Yorkshire and The Humber,0.2,-0.4,-0.1
7,East Midlands,0.1,-0.3,-0.2
8,South West,-0.1,-0.8,-0.6


In [105]:
with pd.ExcelWriter(export_path / 'BSD_entry_exit.xlsx', engine='openpyxl') as writer:
    entry_exit_total.to_excel(writer, sheet_name='headline', index=False)
    entry_exit_size.to_excel(writer, sheet_name='size', index=True)
    exit_age.to_excel(writer, sheet_name='age', index=True)
    entry_exit_prod.to_excel(writer, sheet_name='prod', index=True)
    entry_exit_region.to_excel(writer, sheet_name='region', index=True)
    entry_exit_sector.to_excel(writer, sheet_name='sector', index=True)
    net_entry_region.to_excel(writer, sheet_name='net_region', index=True)

### **SURVIVAL RATES ACROSS COHORTS**

In [97]:
# SURVIVAL RATES BY COHORT - MILESTONE AGES

milestone_ages = [1, 3, 5, 10]

cohorts = sorted(cohort_df['cohort'].unique())

rows = []
for cohort in cohorts:
    df_cohort = cohort_df[cohort_df['cohort'] == cohort].set_index('age')
    
    row = {'Entry cohort': cohort}
    for age in milestone_ages:
        if age in df_cohort.index:
            row[f'age_{age}'] = round(df_cohort.loc[age, 'kaplan_meier_rate'] * 100, 1)
        else:
            row[f'age_{age}'] = None  # cohort hasn't reached this age yet
    
    rows.append(row)

survival_table = pd.DataFrame(rows).set_index('Entry cohort')

survival_table.columns = [f'Age {age}' for age in milestone_ages]

survival_table = survival_table.reset_index()
survival_table = survival_table[survival_table['Entry cohort'] != 2022]
survival_table

survival_table.to_csv(export_path / 'survival_rates_by_cohort.csv', index=False)

### **GROWTH RATES ACROSS COHORTS**

I need to do some treatment of micro firms here. The bimodal distribution is introducing too much volatility in year-to-year growth rates.

### **Reallocation rate tables**